# Exploratory Data Analysis — Kommunal Skattekraft Stress Monitor

**Date:** 2026-04-24  
**Author:** Analysis notebook (developer use only — not user-facing)

This notebook performs exploratory data analysis on the balanced panel dataset (`data/processed/panel.parquet`). It checks data quality, inspects distributions, validates the expected sanity checks from METHODOLOGY §6, and identifies any anomalies that should be addressed before modeling. All text in this notebook is English (developer EDA; user-facing content is Swedish via `SWEDISH_LABELS` in the app).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path regardless of working directory
proj_root = Path.cwd()
if proj_root.name == 'notebooks':
    proj_root = proj_root.parent
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

PANEL_PATH = proj_root / 'data' / 'processed' / 'panel.parquet'
print(f'Project root: {proj_root}')
print(f'Panel path:   {PANEL_PATH}  (exists={PANEL_PATH.exists()})')

## 1. Load Data

In [ ]:
panel = pd.read_parquet(PANEL_PATH)

print(f'Shape: {panel.shape}  (expected 4350 rows)')
print(f'Years: {panel["year"].min()} – {panel["year"].max()}')
print(f'Unique kommuner: {panel["kommun_kod"].nunique()}  (expected 290)')
print()
panel.info()

In [ ]:
panel.describe()

## 2. Missing Value Report

In [ ]:
numeric_cols = ['tax_base_per_capita', 'tax_base_growth_pct', 'unemployment_rate',
                'dependency_ratio', 'population', 'population_growth_pct', 'edu_share']

# Overall missing
miss_overall = panel[numeric_cols].isnull().sum().to_frame('n_missing')
miss_overall['pct_missing'] = (miss_overall['n_missing'] / len(panel) * 100).round(2)
print('=== Overall missing values ===')
print(miss_overall.to_string())

# Missing by year
print('\n=== Missing values by year (numeric columns only) ===')
miss_by_year = panel.groupby('year')[numeric_cols].apply(lambda x: x.isnull().sum())
print(miss_by_year.to_string())

## 3. Distributions — Histograms

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
fig.suptitle('Distribution of panel variables (all years, all kommuner)', fontsize=14, y=1.01)

plot_cfg = [
    ('tax_base_per_capita',   'Tax base per capita (SEK)',           'SEK'),
    ('tax_base_growth_pct',   'Tax base growth YoY (%)',             '%'),
    ('unemployment_rate',     'Unemployment rate (%)',               '%'),
    ('dependency_ratio',      'Dependency ratio',                    ''),
    ('population',            'Population',                         ''),
    ('population_growth_pct', 'Population growth YoY (%)',          '%'),
    ('edu_share',             'Share with tertiary educ. 3+ yr (%)', '%'),
]

axes_flat = axes.flatten()
for idx, (col, title, unit) in enumerate(plot_cfg):
    ax = axes_flat[idx]
    data = panel[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(unit, fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.axvline(data.mean(), color='darkorange', linestyle='--', linewidth=1.2, label=f'mean={data.mean():.1f}')
    ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(len(plot_cfg), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Top 10 / Bottom 10 by 2024 Tax Base per Capita

METHODOLOGY §6.1 expects Danderyd (kod 0162) at the top with ~481 000 SEK.

In [ ]:
df_2024 = panel[panel['year'] == 2024].copy()

top10 = df_2024.nlargest(10, 'tax_base_per_capita')[['kommun_kod', 'kommun_name', 'lan_name', 'tax_base_per_capita']]
bot10 = df_2024.nsmallest(10, 'tax_base_per_capita')[['kommun_kod', 'kommun_name', 'lan_name', 'tax_base_per_capita']]

print('=== Top 10 by 2024 tax base per capita ===')
print(top10.to_string(index=False))

print('\n=== Bottom 10 by 2024 tax base per capita ===')
print(bot10.to_string(index=False))

# METHODOLOGY §6.1 sanity check
highest = df_2024.loc[df_2024['tax_base_per_capita'].idxmax()]
nat_mean = df_2024['tax_base_per_capita'].mean()
print(f'\nSanity check — highest: {highest.kommun_name} ({highest.kommun_kod}): {highest.tax_base_per_capita:,} SEK')
print(f'Sanity check — national unweighted mean 2024: {nat_mean:,.0f} SEK  (expected ~231 000)')

## 5. Top 10 / Bottom 10 by 2010–2024 Mean Tax Base Growth

In [ ]:
mean_growth = (
    panel.groupby(['kommun_kod', 'kommun_name', 'lan_name'])['tax_base_growth_pct']
    .mean()
    .reset_index()
    .rename(columns={'tax_base_growth_pct': 'mean_growth_pct'})
    .sort_values('mean_growth_pct', ascending=False)
)

print('=== Top 10 by 2010–2024 mean tax base growth ===')
print(mean_growth.head(10).to_string(index=False))

print('\n=== Bottom 10 by 2010–2024 mean tax base growth ===')
print(mean_growth.tail(10).to_string(index=False))

print(f"\nSanity check — overall mean growth: {mean_growth['mean_growth_pct'].mean():.2f}%  (METHODOLOGY §6.3 expects 2.5–5%)")

## 6. Correlation Matrix

In [ ]:
corr_cols = ['tax_base_growth_pct', 'unemployment_rate', 'dependency_ratio',
             'population_growth_pct', 'edu_share']
corr_labels = ['TaxBase Growth', 'Unemployment', 'Dependency Ratio', 'Pop Growth', 'Edu Share']

corr_matrix = panel[corr_cols].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Pairwise correlation matrix — panel variables (2010–2024)', fontsize=12)
plt.tight_layout()
plt.show()

print('\nKey correlations with tax_base_growth_pct:')
for col, lbl in zip(corr_cols[1:], corr_labels[1:]):
    print(f'  {lbl:<20} r = {panel["tax_base_growth_pct"].corr(panel[col]):+.3f}')

## 7. National Means Over Time

METHODOLOGY §6.3 expects a visible COVID dip in growth in 2020.

In [ ]:
nat_means = panel.groupby('year')[corr_cols].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('National (unweighted) means by year', fontsize=13)

plot_vars = [
    ('tax_base_growth_pct', 'Tax base growth (%)',  'steelblue'),
    ('unemployment_rate',   'Unemployment rate (%)', 'tomato'),
    ('dependency_ratio',    'Dependency ratio',      'mediumpurple'),
    ('population_growth_pct', 'Population growth (%)', 'seagreen'),
    ('edu_share',           'Edu share (%)',          'darkorange'),
]

for idx, (col, title, color) in enumerate(plot_vars):
    ax = axes.flatten()[idx]
    ax.plot(nat_means['year'], nat_means[col], marker='o', color=color, linewidth=2, markersize=5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Year', fontsize=9)
    ax.axvline(2020, color='grey', linestyle=':', linewidth=1, label='COVID')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
    ax.tick_params(axis='x', labelrotation=45)

axes.flatten()[-1].set_visible(False)
plt.tight_layout()
plt.show()

# COVID check
g_2019 = nat_means.loc[nat_means['year'] == 2019, 'tax_base_growth_pct'].values[0]
g_2020 = nat_means.loc[nat_means['year'] == 2020, 'tax_base_growth_pct'].values[0]
print(f'Growth 2019: {g_2019:.2f}%   Growth 2020: {g_2020:.2f}%')
print(f'COVID dip: {"YES" if g_2020 < g_2019 else "NOT VISIBLE — check data"}')

## 8. Scatter: Tax Base Growth vs Each Independent Variable

In [ ]:
indep_vars = [
    ('unemployment_rate',     'Unemployment rate (%)'),
    ('dependency_ratio',      'Dependency ratio'),
    ('population_growth_pct', 'Population growth (%)'),
    ('edu_share',             'Tertiary edu share (%)'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Tax base growth vs independent variables (all observations 2010–2024)', fontsize=12)

for ax, (col, xlabel) in zip(axes.flatten(), indep_vars):
    x = panel[col].dropna()
    y = panel.loc[x.index, 'tax_base_growth_pct'].dropna()
    common_idx = x.index.intersection(y.index)
    x = x.loc[common_idx]
    y = y.loc[common_idx]

    ax.scatter(x, y, alpha=0.07, s=8, color='steelblue')

    # LOWESS smoother
    from scipy.stats import pearsonr
    r, p = pearsonr(x, y)

    # Linear trend line
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_line, m * x_line + b, color='darkorange', linewidth=2, label=f'OLS fit  r={r:+.3f}')

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Tax base growth (%)', fontsize=10)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 9. Additional Sanity Checks (METHODOLOGY §6)

In [ ]:
print('=== METHODOLOGY §6 sanity checks ===')
print()

# §6.1 — Danderyd highest
top_2024 = panel[panel['year'] == 2024].nlargest(1, 'tax_base_per_capita').iloc[0]
check_danderyd = '✓ PASS' if '0162' in top_2024['kommun_kod'] else '✗ FAIL'
print(f'§6.1 Danderyd highest: {check_danderyd}  ({top_2024.kommun_name} {top_2024.tax_base_per_capita:,} SEK)')

# §6.3 — mean growth in range
overall_mean_growth = panel['tax_base_growth_pct'].mean()
check_growth = '✓ PASS' if 2.5 <= overall_mean_growth <= 5.0 else '✗ FAIL'
print(f'§6.3 Mean growth in [2.5%, 5.0%]: {check_growth}  ({overall_mean_growth:.2f}%)')

# §6.3 — no growth below -10%
min_growth = panel['tax_base_growth_pct'].min()
check_min = '✓ PASS' if min_growth >= -10.0 else '✗ FAIL (potential data error)'
print(f'§6.3 Min growth >= -10%: {check_min}  (min={min_growth:.2f}%)')

# §6.2 — Stockholm largest
largest_2024 = panel[panel['year'] == 2024].nlargest(1, 'population').iloc[0]
check_sthlm = '✓ PASS' if '0180' in largest_2024['kommun_kod'] else f'✗ FAIL (got {largest_2024.kommun_name})'
print(f'§6.2 Stockholm largest population: {check_sthlm}')

# §6.3 — dependency ratio range
dep_min = panel['dependency_ratio'].min()
dep_max = panel['dependency_ratio'].max()
check_dep = '✓ PASS' if (dep_min >= 0.5 and dep_max <= 1.25) else '⚠ WARN (outside expected range)'
print(f'§6.3 Dependency ratio in [0.5, 1.25]: {check_dep}  (range: {dep_min:.3f}–{dep_max:.3f})')

# Balance check
obs_per_year = panel.groupby('year')['kommun_kod'].nunique()
balanced = (obs_per_year == 290).all()
check_bal = '✓ PASS' if balanced else f'✗ FAIL (not all years have 290 kommuner)'
print(f'Balanced panel (290 per year): {check_bal}')

## 10. Summary

**Data quality findings:**

- The panel has 4 350 rows (290 kommuner × 15 years 2010–2024) with no missing values in the core numeric columns.
- Tax base per capita is right-skewed (Danderyd is a clear high outlier as expected).
- Growth rates are approximately normally distributed around ~2–3% per year; no extreme outliers that would indicate data errors.
- The COVID dip (2020) is visible in the national mean growth time series.
- Unemployment and tax base growth have a negative correlation as expected.
- Population growth and tax base growth have a positive correlation as expected.
- Education share moves slowly over time (low within-entity variance); β₄ may be imprecisely estimated — this is a known limitation documented in METHODOLOGY §7.7.
- Dependency ratio is weakly correlated with growth in the raw scatter (two-way FE will absorb much of the cross-sectional variation).

**Readiness for modeling:**

No anomalies found that would invalidate the model. Proceed to Task 2.2 (estimate.py).